# xMICD: Example Usage Notebook

This notebook demonstrates how to use the `xMICD` class to:

- Load ICD code embeddings  
- Load the hierarchical grouping structure  
- Build the xMICD model from these inputs  
- Retrieve vector representations for individual ICD codes or aggregated sets of codes  

Each section introduces one step of the workflow and can be adapted to your own data.


## 1. Set up

We begin by importing the required libraries and loading the ICD embedding matrix.

In [1]:
from xMICD import xMICD
import pandas as pd

### Load embedding matrix

The file `icd_code_vec.csv` contains pre-trained embeddings for ICD codes, typically one vector per code.

The embedding file is loaded into `embedding_df`.  
Each row corresponds to one ICD code, and the remaining columns store its embedding dimensions.
The columns are renamed with the prefix `Embedding_` to match the expected format of `xMICD`.


In [2]:
embedding_df = pd.read_csv("Input/icd_code_vec.csv")
embedding_df.columns = ["DIS_CODE"]+["Embedding_"+col for col in embedding_df.columns.to_list()[1:]]
embedding_df.head()

,DIS_CODE,Embedding_0,Embedding_1,Embedding_2,Embedding_3,Embedding_4,Embedding_5,Embedding_6,Embedding_7,Embedding_8,...,Embedding_1014,Embedding_1015,Embedding_1016,Embedding_1017,Embedding_1018,Embedding_1019,Embedding_1020,Embedding_1021,Embedding_1022,Embedding_1023
0,A00,0.062662,0.069492,-0.062197,0.027203,-0.122601,0.109749,0.033870,0.011480,-0.158279,...,-0.181965,0.217679,-0.039114,-0.112860,0.220006,-0.119871,0.099518,-0.230879,-0.325332,-0.109664
1,A000,0.067028,0.110111,-0.009773,0.027653,-0.139956,0.113899,-0.014510,-0.056627,-0.092039,...,-0.112203,0.253481,-0.001807,-0.079261,0.216512,-0.110161,0.033945,-0.228332,-0.350511,-0.175411
2,A001,0.012806,-0.073449,-0.194383,0.030890,-0.104823,0.071724,0.255976,0.289722,-0.450981,...,-0.485810,0.057378,-0.200740,-0.266606,0.282643,-0.191786,0.387447,-0.246718,-0.278767,0.134742
3,A009,0.087436,0.119059,-0.020660,0.023321,-0.124710,0.130990,-0.052174,-0.094882,-0.045424,...,-0.065448,0.277363,0.018373,-0.053212,0.190165,-0.089956,-0.014772,-0.222431,-0.332015,-0.198793
4,A01,-0.166250,0.126540,-0.080693,-0.025942,-0.008737,0.047102,-0.038586,0.383638,-0.240688,...,-0.284910,0.111417,0.022381,-0.168525,0.064540,-0.158785,0.043830,-0.392468,-0.507154,0.007472


### Load grouping file

The file `grouping.csv` defines the hierarchical relationships between ICD codes (e.g. parent–child groups).
xMICD uses this structure to construct anchor (group-level) embeddings.


In [3]:
grouping_df = pd.read_csv("Input/grouping.csv")
grouping_df.head()

,ICD_code,Group
0,A00,A00-A09
1,A000,A00-A09
2,A001,A00-A09
3,A009,A00-A09
4,A01,A00-A09


## 3. Build the xMICD model

We now initialize `xMICD` and build the hierarchical embedding structure from:

- the ICD embedding matrix (`embedding_df`)  
- the grouping table (`grouping_df`)  

The argument `method` in `xMICD.build_from_grouping` controls how anchor (group-level) embeddings are constructed:

- `method="similarity"` (default): for each group, selects the ICD code whose embedding has the highest average cosine similarity to other codes in the same group (a “medoid”).  
- `method="mean"`: for each group, computes the anchor as the mean of all available ICD embeddings in that group.  

After this step, the model is ready to answer ICD vector queries.


In [4]:
xMICD = xMICD()
xMICD.build_from_grouping(
  embedding_df=embedding_df,
  grouping_df=grouping_df,
  embeddings_icd_column="DIS_CODE",
  embeddings_embedding_column_prefix="Embedding_",
  groupings_icd_column="ICD_code",
  groupings_group_column="Group",
  method="similarity"
)

You can extract and inspect the anchor table constructed by xMICD.
Anchors represent group-level embeddings derived from the hierarchical ICD structure.

In [5]:
anchor_df = xMICD.get_anchors_df()
anchor_df.head()

,ICD_code,Embedding_0,Embedding_1,Embedding_2,Embedding_3,Embedding_4,Embedding_5,Embedding_6,Embedding_7,Embedding_8,...,Embedding_1014,Embedding_1015,Embedding_1016,Embedding_1017,Embedding_1018,Embedding_1019,Embedding_1020,Embedding_1021,Embedding_1022,Embedding_1023
0,A00-A09,0.016789,0.010529,0.125331,0.010457,-0.198151,0.001793,0.044448,0.181032,-0.417463,...,-0.097866,0.353959,0.051239,-0.071135,0.159027,-0.136780,0.099153,-0.368199,-0.291492,-0.035028
1,A15-A19,0.064626,-0.008522,0.063888,0.037330,-0.014776,0.146857,-0.257068,0.240926,-0.219253,...,-0.100309,0.090913,0.010420,0.084898,0.115204,-0.125848,-0.139514,-0.427525,-0.330940,-0.168843
2,A20-A28,-0.045135,0.073171,0.048047,0.082945,-0.095362,-0.014199,-0.025004,0.413631,-0.218214,...,-0.094656,0.094021,-0.153695,-0.075135,-0.027920,-0.095387,0.085441,-0.295191,-0.304068,0.057431
3,A30-A49,-0.058216,0.043455,-0.108780,0.113253,-0.067487,0.094091,-0.012400,0.221222,-0.155525,...,-0.154410,0.142775,0.042463,-0.160190,-0.001310,0.021940,0.084887,-0.333547,-0.397973,-0.054987
4,A50-A64,-0.015272,0.018028,-0.222336,-0.006442,-0.149869,0.061234,-0.012696,0.417944,-0.105432,...,-0.133328,0.165330,0.051859,-0.155567,0.251349,0.142224,-0.071994,-0.383247,-0.322037,0.053474


You may save this table and later rebuild xMICD using `build_from_anchor` to save time in larger workflows.

In [6]:
# xMICD = xMICD()
# xMICD.build_from_anchor(
#     embedding_df = embedding_df,
#     anchor_df = anchor_df,
#     embeddings_icd_column = "DIS_CODE",
#     embeddings_embedding_column_prefix = "Embedding_",
#     anchor_icd_column = "ICD_code",
#     anchor_embedding_column_prefix = "Embedding_"
# )

# 4. Usage examples

The following sections show how to query vectors from the xMICD model under different settings.


## ICD vectors without parent reduction

Here we request vectors for a list of ICD codes directly, without using parent reduction.
The model returns:

- `valid_icds`: the subset of input codes that can be mapped  
- `icd_vectors`: the corresponding embedding vectors  


In [7]:
valid_icds, icd_vectors = xMICD.get_icd_vector(["A00","B03","S52001A"])

C:\Users\Lenovo\Downloads\Thesis\ทุน FF\xMICS_Demo\xMICS_Demo_Kup_s Edition\xMICD.py:124: UserWarning: ICD S52001A does not exist in embedding database. Skipped ICD S52001A
  warnings.warn(f"ICD {icd} does not exist in embedding database. Skipped ICD {icd}")


We first request the vector representation of each ICD code separately.

In [8]:
#Give the embedding of each ICD code separately
print("Valid ICD codes: "+" ".join(valid_icds))
print("ICD vectors: ")
print(icd_vectors)

Valid ICD codes: A00 B03
ICD vectors: 
[[0.76249668 0.5678507  0.59392934 1.         0.51789735 0.92361997
  0.55806533 0.78811792 0.92090353 0.87267894 0.7562767  0.59087534
  0.88478868 0.68076455 0.61748728 0.76304769 0.62039575 0.48757289
  0.29011991 0.98748173 0.7157038  0.58948968 0.61893986 0.58625036
  0.51308563 0.50084729 0.55601145 0.45270674 0.71418723 0.71403595
  0.60537056 0.7015945  0.59567914 0.55206741 0.47906068 0.84621332
  0.88891511 0.46848178 0.71576588 0.76163385 0.89418124 0.66567131
  0.41211194 0.81579005 0.43318171 0.76226572 0.62016796 0.79140219
  0.14258626 0.62041221 0.77235459 0.89133591 0.68430404 0.55914748
  0.67881969 0.75264305 0.65497413 0.55558041 0.55798697 0.51851534
  0.6551425  0.56229184 0.94746436 0.33701658 0.4448423  0.3580515
  0.41112319 0.56417974 0.50513608 0.54515497 0.63848294 0.33472257
  0.56692855 0.71825181 0.35604883 0.18559713 0.57710723 0.62960092
  0.29205394 0.0543976  0.25910209 0.49342097 0.39466101 0.87499305
  0.219553

### Aggregated embedding

Here we compute a single aggregated vector for the set of ICD codes using the `"max"` aggregation method.

The argument `method` in `xMICD.get_aggregated_vector` controls how individual xMICD vectors are combined:

- `method="max"` (default): elementwise maximum over all ICD vectors.  
- `method="avg"`: elementwise mean over all ICD vectors.  
- `method="avg3top"`: for each dimension, takes the average of the three largest values across ICD vectors.  

You can choose the aggregation strategy depending on whether you want a more “any-code-activates” profile (`max`) or a smoother, averaged profile (`avg` / `avg3top`).


In [9]:
valid_icds, icd_vectors = xMICD.get_aggregated_vector(["A00","B03","S52001A"],method="max")

C:\Users\Lenovo\Downloads\Thesis\ทุน FF\xMICS_Demo\xMICS_Demo_Kup_s Edition\xMICD.py:124: UserWarning: ICD S52001A does not exist in embedding database. Skipped ICD S52001A
  warnings.warn(f"ICD {icd} does not exist in embedding database. Skipped ICD {icd}")


In [10]:
#Give only one aggregated embedding of all ICD codes
print("Valid ICD codes: "+" ".join(valid_icds))
print("Aggregated ICD vector: ")
print(icd_vectors)

Valid ICD codes: A00 B03
Aggregated ICD vector: 
[0.76249668 0.62084617 0.59392934 1.         0.5625012  0.92361997
 0.55806533 0.78811792 0.96044763 0.92089031 0.88411885 0.59177453
 1.         0.70277346 0.61748728 0.76304769 0.62039575 0.48757289
 0.40997846 0.98748173 0.7157038  0.58948968 0.65351487 0.58625036
 0.51308563 0.50084729 0.55601145 0.45270674 0.71418723 0.75750212
 0.60806725 0.77524833 0.62672683 0.55206741 0.47906068 0.84621332
 0.91787875 0.46848178 0.7303615  0.76163385 0.89418124 0.66567131
 0.41211194 0.81579005 0.43318171 0.76226572 0.62016796 0.81990993
 0.14258626 0.64800194 0.77235459 0.89133591 0.68430404 0.58403413
 0.68829751 0.79751257 0.69199392 0.56903615 0.55798697 0.52281741
 0.65986861 0.56229184 0.94746436 0.33701658 0.4448423  0.3580515
 0.4339526  0.56417974 0.50513608 0.55749158 0.64960657 0.33472257
 0.56692855 0.71825181 0.35604883 0.20110586 0.57710723 0.6487274
 0.29205394 0.0543976  0.25910209 0.49342097 0.46927929 0.87499305
 0.21955314 0.5

## Using `use_parent_embedding=True`

When `use_parent_embedding=True`, xMICD can reduce very specific ICD codes (e.g. `A008`)
to a valid parent code (e.g. `A00`) before computing their vectors.

This is useful when:

- some detailed codes are rare or missing  
- you want more stable, higher-level representations of ICD patterns  


In [11]:
valid_icds, icd_vectors = xMICD.get_icd_vector(["A00","B03","S52001A"],use_parent_embedding=True)

In [12]:
#With use_parent_embedding, S52001A could be reduced to S520
print("Valid ICD codes: "+" ".join(valid_icds))
print("ICD vectors: ")
print(icd_vectors)

Valid ICD codes: A00 B03 S52001A
ICD vectors: 
[[0.76249668 0.5678507  0.59392934 1.         0.51789735 0.92361997
  0.55806533 0.78811792 0.92090353 0.87267894 0.7562767  0.59087534
  0.88478868 0.68076455 0.61748728 0.76304769 0.62039575 0.48757289
  0.29011991 0.98748173 0.7157038  0.58948968 0.61893986 0.58625036
  0.51308563 0.50084729 0.55601145 0.45270674 0.71418723 0.71403595
  0.60537056 0.7015945  0.59567914 0.55206741 0.47906068 0.84621332
  0.88891511 0.46848178 0.71576588 0.76163385 0.89418124 0.66567131
  0.41211194 0.81579005 0.43318171 0.76226572 0.62016796 0.79140219
  0.14258626 0.62041221 0.77235459 0.89133591 0.68430404 0.55914748
  0.67881969 0.75264305 0.65497413 0.55558041 0.55798697 0.51851534
  0.6551425  0.56229184 0.94746436 0.33701658 0.4448423  0.3580515
  0.41112319 0.56417974 0.50513608 0.54515497 0.63848294 0.33472257
  0.56692855 0.71825181 0.35604883 0.18559713 0.57710723 0.62960092
  0.29205394 0.0543976  0.25910209 0.49342097 0.39466101 0.87499305
  

### Aggregated embedding with parent reduction

Finally, we compute an aggregated embedding using the `"max"` method and enabling parent reduction.
This produces a stable representation for the set of ICD codes after mapping them to valid parents.


In [13]:
valid_icds, icd_vectors = xMICD.get_aggregated_vector(["A00","B03","S52001A"],use_parent_embedding=True,method="max")

In [14]:
#With use_parent_embedding, S52001A could be reduced to S520
print("Valid ICD codes: "+" ".join(valid_icds))
print("Aggregated vector: ")
print(icd_vectors)

Valid ICD codes: A00 B03 S52001A
Aggregated vector: 
[0.76249668 0.62084617 0.59392934 1.         0.5625012  0.92361997
 0.55806533 0.78811792 0.96044763 0.92089031 0.88411885 0.59177453
 1.         0.70277346 0.61748728 0.76304769 0.62039575 0.48757289
 0.43475976 0.98748173 0.7157038  0.58948968 0.65351487 0.58625036
 0.60538837 0.54494782 0.55601145 0.45270674 0.71418723 0.75750212
 0.60806725 0.77524833 0.62672683 0.55206741 0.47906068 0.84621332
 0.91787875 0.46848178 0.7303615  0.76163385 0.89418124 0.66567131
 0.48259359 0.81579005 0.43318171 0.76226572 0.62016796 0.81990993
 0.14258626 0.64800194 0.77235459 0.89133591 0.68430404 0.58403413
 0.68829751 0.79751257 0.69199392 0.6109834  0.61441356 0.52281741
 0.65986861 0.61336447 0.94746436 0.61540217 0.49585284 0.5472544
 0.64186151 0.56417974 0.59048063 0.6635535  0.64960657 0.46201896
 0.56692855 0.71825181 0.51481851 0.46948559 0.57710723 0.68329767
 0.33829016 0.0543976  0.25910209 0.55529175 0.51931223 0.87499305
 0.5502830